# GemmaFit v3 LiteRT-LM Conversion Matrix

Conversion-only Colab notebook for the already-trained GemmaFit v3 Evidence Router.

This notebook **does not train** and **does not import Unsloth**. It starts from the merged HF/safetensors export on Google Drive, converts several `.litertlm` variants, records resumable state/events to Drive, and packages per-variant download bundles for Windows/Pixel smoke testing.

Current Android failure being debugged:

- Existing `dynamic_wi4_afp32 + cache_length=2048 + prefill=256` artifact is readable on Pixel 8 Pro.
- GPU and CPU both fail at `engine_initialize` in LiteRT-LM.
- We need conversion variants to find the first artifact that can create an Android engine.

Default behavior: run `wi8_cache1024_prefill128` and then the faster backup `wi8_cache512_prefill64`. Existing artifacts are skipped unless `FORCE_EXPORT=True`, so rerunning after the first export will only produce missing backups.

## 0. Mount Drive, define paths, and print resume diagnostic

Run this first after every Colab reconnect. It finds the merged HF export and prints completed variants, missing variants, and the exact Windows finalize commands expected after download.

In [ ]:
from google.colab import drive
import glob
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

# --- User config -----------------------------------------------------------
DRIVE_ROOT = Path('/content/drive/MyDrive/GemmaFit_train')
TRAINED_OUTPUT_DIR = DRIVE_ROOT / 'trained_outputs'
RUN_SUFFIX = 'gemmafit_v3_evidence_router'
DEFAULT_RUN_NAME = 'unsloth_gemma-4-E4B-it_gemmafit_v3_evidence_router'
ARTIFACT_PREFIX = 'gemmafit-v3-evidence-router'

# Default: export the current best candidate plus a smaller backup.
# If wi8/cache1024 already exists, rerun will skip it and only produce wi8/cache512.
# To run the full matrix, set:
# RUN_VARIANTS = ['wi8_cache1024_prefill128', 'wi8_cache512_prefill64', 'wi4_cache1024_prefill128', 'wi8_cache2048_prefill256']
RUN_VARIANTS = ['wi8_cache1024_prefill128', 'wi8_cache512_prefill64']
FORCE_EXPORT = False
STOP_AFTER_FIRST_SUCCESS = False

VARIANTS = [
    {
        'name': 'wi8_cache1024_prefill128',
        'quantization_recipe': 'dynamic_wi8_afp32',
        'cache_length': 1024,
        'prefill_lengths': [128],
        'note': 'Recommended first Android smoke candidate: conservative quantization and smaller cache.',
    },
    {
        'name': 'wi8_cache512_prefill64',
        'quantization_recipe': 'dynamic_wi8_afp32',
        'cache_length': 512,
        'prefill_lengths': [64],
        'note': 'Fast backup candidate if cache1024 still fails Android engine creation.',
    },
    {
        'name': 'wi4_cache1024_prefill128',
        'quantization_recipe': 'dynamic_wi4_afp32',
        'cache_length': 1024,
        'prefill_lengths': [128],
        'note': 'Smaller Q4 candidate after wi8/cache1024 proves engine creation.',
    },
    {
        'name': 'wi8_cache2048_prefill256',
        'quantization_recipe': 'dynamic_wi8_afp32',
        'cache_length': 2048,
        'prefill_lengths': [256],
        'note': 'Closer to original context length, but higher RAM/converter pressure.',
    },
]
VARIANTS_BY_NAME = {v['name']: v for v in VARIANTS}

EXPORT_MATRIX_DIR = DRIVE_ROOT / 'litert_export_v3_matrix'
LOCAL_WORK_ROOT = Path('/content/gemmafit_litert_matrix_work')
RUN_STATE = TRAINED_OUTPUT_DIR / f'RUN_STATE_{DEFAULT_RUN_NAME}_litert_matrix.json'
RUN_EVENTS = TRAINED_OUTPUT_DIR / f'RUN_EVENTS_{DEFAULT_RUN_NAME}_litert_matrix.jsonl'
DISCONNECT_POINTS = TRAINED_OUTPUT_DIR / f'DISCONNECT_POINTS_{DEFAULT_RUN_NAME}_litert_matrix.jsonl'
MATRIX_SUMMARY = TRAINED_OUTPUT_DIR / f'LITERT_MATRIX_{DEFAULT_RUN_NAME}.json'
DONE_MARKER = TRAINED_OUTPUT_DIR / f'TRAINING_DONE_{DEFAULT_RUN_NAME}.json'

# --- Helpers ---------------------------------------------------------------
def now_iso():
    return time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass
    tmp.replace(path)

def append_jsonl(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(payload, ensure_ascii=False) + '\n')
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass

def record_event(phase, event, **kwargs):
    payload = {'ts': now_iso(), 'phase': phase, 'event': event, **kwargs}
    append_jsonl(RUN_EVENTS, payload)
    if event in {'variant_start', 'export_success', 'export_failed', 'bundle_written', 'safe_resume_point'}:
        append_jsonl(DISCONNECT_POINTS, payload)
    return payload

def write_run_state(**kwargs):
    state = {'updated_at': now_iso(), 'run_name': DEFAULT_RUN_NAME, 'run_suffix': RUN_SUFFIX, **kwargs}
    atomic_write_json(RUN_STATE, state)
    return state

def file_sha256(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def has_nonzero_safetensors(path):
    files = list(Path(path).rglob('*.safetensors'))
    return bool(files) and all(p.stat().st_size > 0 for p in files)

def find_merged_path():
    explicit = TRAINED_OUTPUT_DIR / 'merged_fp16' / f'{DEFAULT_RUN_NAME}_merged_fp16'
    candidates = []
    if explicit.exists():
        candidates.append(explicit)
    candidates.extend(Path(p) for p in glob.glob(str(TRAINED_OUTPUT_DIR / 'merged_fp16' / f'*_{RUN_SUFFIX}_merged_fp16')))
    candidates = [p for p in candidates if p.exists() and has_nonzero_safetensors(p)]
    candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f'No valid merged HF export found under {TRAINED_OUTPUT_DIR / "merged_fp16"}')
    return candidates[0]

def variant_paths(name):
    drive_dir = EXPORT_MATRIX_DIR / name
    local_dir = LOCAL_WORK_ROOT / name
    final_litertlm = DRIVE_ROOT / f'{ARTIFACT_PREFIX}-{name}.litertlm'
    bundle_zip = DRIVE_ROOT / f'{ARTIFACT_PREFIX}-{name}-local-artifacts.zip'
    log_path = EXPORT_MATRIX_DIR / f'{ARTIFACT_PREFIX}-{name}-export.log'
    return {
        'drive_dir': drive_dir,
        'local_dir': local_dir,
        'final_litertlm': final_litertlm,
        'bundle_zip': bundle_zip,
        'log_path': log_path,
    }

def find_litertlm(export_dir):
    files = list(Path(export_dir).rglob('*.litertlm'))
    if not files:
        return None
    return max(files, key=lambda p: (p.stat().st_size, p.stat().st_mtime))

def existing_variant_status(name):
    paths = variant_paths(name)
    artifact = paths['final_litertlm']
    if artifact.exists() and artifact.stat().st_size > 0:
        return {'name': name, 'status': 'exists', 'path': str(artifact), 'size_bytes': artifact.stat().st_size}
    return {'name': name, 'status': 'missing', 'path': str(artifact), 'size_bytes': 0}

# --- Mount and diagnostic --------------------------------------------------
drive.mount('/content/drive', force_remount=False)
MERGED_PATH = find_merged_path()
EXPORT_MATRIX_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)

selected = []
for name in RUN_VARIANTS:
    if name not in VARIANTS_BY_NAME:
        raise ValueError(f'Unknown RUN_VARIANTS entry: {name}. Valid: {list(VARIANTS_BY_NAME)}')
    selected.append(VARIANTS_BY_NAME[name])

statuses = [existing_variant_status(v['name']) for v in VARIANTS]
write_run_state(
    phase='0_resume_diagnostic',
    merged_path=str(MERGED_PATH),
    run_variants=RUN_VARIANTS,
    variants=statuses,
)
record_event('0_resume_diagnostic', 'safe_resume_point', merged_path=str(MERGED_PATH), run_variants=RUN_VARIANTS)

print('=== Resume diagnostic ===')
print('Merged HF export:', MERGED_PATH)
print('Drive export matrix dir:', EXPORT_MATRIX_DIR)
print('Local converter work root:', LOCAL_WORK_ROOT)
print('Run variants:', RUN_VARIANTS)
print('\nVariant status:')
for status in statuses:
    print('-', status['name'], status['status'], status['size_bytes'], status['path'])
print('\nWindows finalize command template after downloading a per-variant zip:')
print(r'cd /d D:\GemmaFit')
print(r'python finetune\prepare_litert_artifact.py --source-bundle "C:\path\to\gemmafit-v3-evidence-router-<variant>-local-artifacts.zip" --run-smoke')

## 1. Install conversion dependencies

Run once in a fresh conversion-only runtime. If this installs or upgrades Torch / TorchAO / Transformers, restart the runtime, then rerun cell 0 and cell 2.

Do **not** import Unsloth in this notebook.

In [ ]:
import os
import subprocess
import sys

INSTALL_LITERT_DEPS = True
UPGRADE_TORCH_NIGHTLY = True
AUTO_RESTART_AFTER_INSTALL = False

if INSTALL_LITERT_DEPS:
    cmd = [
        sys.executable, '-m', 'pip', 'install', '-U', '--pre',
        'litert-torch-nightly', 'litert-lm', 'sentencepiece', 'protobuf',
        'transformers', 'huggingface_hub', 'accelerate', 'safetensors',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

if UPGRADE_TORCH_NIGHTLY:
    # Keep torch and torchao from the same nightly CUDA index. This avoids the
    # torch>=2.11 / torchao.quantization.pt2e import mismatch seen in Colab.
    cmd = [
        sys.executable, '-m', 'pip', 'install', '-U', '--pre',
        'torch', 'torchao',
        '--index-url', 'https://download.pytorch.org/whl/nightly/cu128',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

print('\nInstall finished.')
print('If any loaded package changed, restart runtime, then rerun cell 0 and cell 2.')
if AUTO_RESTART_AFTER_INSTALL:
    print('AUTO_RESTART_AFTER_INSTALL=True; killing runtime now.')
    os.kill(os.getpid(), 9)

## 2. Verify exporter import and environment

This catches the common failure modes before the long conversion starts:

- Torch still below 2.11
- TorchAO missing `torchao.quantization.pt2e.quantize_pt2e`
- `litert_torch.generative.export_hf` import failure
- Fire help returning non-zero but valid help output

In [ ]:
import importlib.util
import re
import subprocess
import sys

print('Python:', sys.version)
try:
    import torch
    torch_version = torch.__version__
    print('torch:', torch_version)
    print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as exc:
    raise RuntimeError('torch import failed; rerun install cell and restart runtime.') from exc

try:
    import torchao
    print('torchao:', getattr(torchao, '__version__', 'unknown'))
except Exception as exc:
    raise RuntimeError('torchao import failed; rerun install cell and restart runtime.') from exc

if importlib.util.find_spec('torchao.quantization.pt2e.quantize_pt2e') is None:
    raise RuntimeError(
        'torchao is present but missing torchao.quantization.pt2e.quantize_pt2e. '
        'Rerun install cell with the nightly torch index, restart runtime, then rerun cells 0 and 2.'
    )

def version_tuple(version):
    match = re.match(r'(\d+)\.(\d+)', version or '')
    return tuple(map(int, match.groups())) if match else (0, 0)

if version_tuple(torch_version) < (2, 11):
    raise RuntimeError(f'torch {torch_version} is too old for litert-torch-nightly; need >=2.11.')

help_cmd = [sys.executable, '-m', 'litert_torch.generative.export_hf', '--help']
help_proc = subprocess.run(help_cmd, text=True, capture_output=True, check=False)
help_text = (help_proc.stdout or '') + (help_proc.stderr or '')
print('export_hf --help returncode:', help_proc.returncode)
print(help_text[:4000])

if 'Please upgrade to torch >= 2.11.0' in help_text:
    raise RuntimeError('litert_torch still sees an old torch. Restart runtime after install.')
if "No module named 'torchao.quantization.pt2e.quantize_pt2e'" in help_text:
    raise RuntimeError('litert_torch still sees incompatible torchao. Reinstall torchao from nightly index and restart.')

help_looks_valid = (
    'SYNOPSIS' in help_text
    and 'POSITIONAL ARGUMENTS' in help_text
    and '--quantization_recipe' in help_text
    and '--prefill_lengths' in help_text
)
if help_proc.returncode != 0 and help_looks_valid:
    print('Valid Fire help with non-zero return code; continuing is OK.')
elif help_proc.returncode != 0:
    raise RuntimeError('LiteRT exporter is not importable. See help output above.')

record_event('2_verify_exporter', 'safe_resume_point', torch=torch_version)
write_run_state(phase='2_verify_exporter', torch=torch_version)

## 3. Convert selected variants

This cell runs only `RUN_VARIANTS` from cell 0. It uses `/content` for converter scratch/output, then copies only finished artifacts back to Drive.

If Colab disconnects, rerun cell 0 and this cell. Existing final artifacts are skipped unless `FORCE_EXPORT=True`.

In [ ]:
import gc
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

EXPORT_TIMEOUT_SEC = None  # Keep None for long conversions; Colab itself may still disconnect.
KEEP_LOCAL_EXPORT_DIR = False

results = []
summary = {'updated_at': now_iso(), 'merged_path': str(MERGED_PATH), 'variants': []}

for variant in selected:
    name = variant['name']
    paths = variant_paths(name)
    final_litertlm = paths['final_litertlm']
    if final_litertlm.exists() and final_litertlm.stat().st_size > 0 and not FORCE_EXPORT:
        print(f'\n=== {name}: existing artifact, skipping export ===')
        result = {
            'name': name,
            'status': 'exists',
            'artifact': str(final_litertlm),
            'size_bytes': final_litertlm.stat().st_size,
            'sha256': file_sha256(final_litertlm),
            'variant': variant,
        }
        results.append(result)
        summary['variants'].append(result)
        continue

    local_dir = paths['local_dir']
    drive_dir = paths['drive_dir']
    log_path = paths['log_path']
    shutil.rmtree(local_dir, ignore_errors=True)
    local_dir.mkdir(parents=True, exist_ok=True)
    drive_dir.mkdir(parents=True, exist_ok=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    prefill_literal = '[' + ','.join(str(x) for x in variant['prefill_lengths']) + ']'
    cmd = [
        sys.executable, '-m', 'litert_torch.generative.export_hf',
        str(MERGED_PATH), str(local_dir),
        f'--prefill_lengths={prefill_literal}',
        f"--cache_length={variant['cache_length']}",
        f"--quantization_recipe={variant['quantization_recipe']}",
        '--externalize_embedder',
        '--use_jinja_template',
    ]

    print(f'\n=== Export {name} ===')
    print('Variant:', variant)
    print('Command:', ' '.join(cmd))
    record_event('3_convert', 'variant_start', variant=name, command=' '.join(cmd), note=variant.get('note', ''))
    write_run_state(phase='3_convert', active_variant=name, command=' '.join(cmd))

    started = time.time()
    with open(log_path, 'w', encoding='utf-8') as log:
        log.write(f'=== {now_iso()} START {name} ===\n')
        log.write(json.dumps(variant, indent=2) + '\n')
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        last_lines = []
        try:
            for line in proc.stdout:
                print(line, end='')
                log.write(line)
                last_lines.append(line.rstrip())
                last_lines = last_lines[-80:]
            return_code = proc.wait(timeout=EXPORT_TIMEOUT_SEC)
        except Exception:
            proc.kill()
            raise
        finally:
            log.flush()

    elapsed = int(time.time() - started)
    artifact = find_litertlm(local_dir)
    if return_code != 0 or artifact is None or artifact.stat().st_size == 0:
        result = {
            'name': name,
            'status': 'export_failed',
            'return_code': return_code,
            'elapsed_seconds': elapsed,
            'log_path': str(log_path),
            'last_lines': last_lines[-30:],
            'variant': variant,
        }
        results.append(result)
        summary['variants'].append(result)
        atomic_write_json(MATRIX_SUMMARY, summary)
        record_event('3_convert', 'export_failed', variant=name, return_code=return_code, elapsed_seconds=elapsed, log_path=str(log_path))
        print(f'\n{name} failed. See log: {log_path}')
        continue

    shutil.copy2(artifact, final_litertlm)
    shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
    size = final_litertlm.stat().st_size
    sha = file_sha256(final_litertlm)
    result = {
        'name': name,
        'status': 'export_success',
        'return_code': return_code,
        'elapsed_seconds': elapsed,
        'artifact': str(final_litertlm),
        'size_bytes': size,
        'sha256': sha,
        'log_path': str(log_path),
        'drive_export_dir': str(drive_dir),
        'variant': variant,
    }
    results.append(result)
    summary['variants'].append(result)
    atomic_write_json(MATRIX_SUMMARY, summary)
    record_event('3_convert', 'export_success', variant=name, elapsed_seconds=elapsed, artifact=str(final_litertlm), size_bytes=size, sha256=sha)
    print(f'\n{name} exported: {final_litertlm}')
    print('size bytes:', size)
    print('sha256:', sha)

    if not KEEP_LOCAL_EXPORT_DIR:
        shutil.rmtree(local_dir, ignore_errors=True)
    gc.collect()
    if STOP_AFTER_FIRST_SUCCESS:
        print('STOP_AFTER_FIRST_SUCCESS=True; stopping after first successful export.')
        break

write_run_state(phase='3_convert_done', results=results, matrix_summary=str(MATRIX_SUMMARY))
print('\n=== Conversion matrix summary ===')
print(json.dumps(summary, indent=2))
print('Summary path:', MATRIX_SUMMARY)

## 4. Package per-variant Windows bundles

Creates one zip per successful variant. Download a zip and run the printed Windows command. The local script will copy that variant to `models/gemmafit-v3-evidence-router.litertlm`, which is what the Android resolver prefers.

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

matrix = json.loads(Path(MATRIX_SUMMARY).read_text(encoding='utf-8')) if Path(MATRIX_SUMMARY).exists() else {'variants': []}
successful = [v for v in matrix.get('variants', []) if v.get('status') in {'export_success', 'exists'} and Path(v.get('artifact', '')).exists()]
if not successful:
    raise RuntimeError('No successful variants to package. Run conversion cell first.')

bundle_results = []
for item in successful:
    name = item['name']
    paths = variant_paths(name)
    artifact = Path(item['artifact'])
    bundle_zip = paths['bundle_zip']
    bundle_root = Path('/content') / f'gemmafit_v3_litert_bundle_{name}'
    shutil.rmtree(bundle_root, ignore_errors=True)

    model_dst = bundle_root / 'models' / f'{ARTIFACT_PREFIX}.litertlm'
    model_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(artifact, model_dst)

    metrics_dir = bundle_root / 'finetune' / 'metrics'
    metrics_dir.mkdir(parents=True, exist_ok=True)
    variant_done = {
        'version': 'v3_evidence_router',
        'run_name': DEFAULT_RUN_NAME,
        'run_suffix': RUN_SUFFIX,
        'status': 'litert_conversion_complete_unverified',
        'litert_variant': name,
        'litertlm_path': 'models/gemmafit-v3-evidence-router.litertlm',
        'source_litertlm': str(artifact),
        'conversion_status': 'converted_unverified',
        'conversion_log': item,
        'matrix_summary': str(MATRIX_SUMMARY),
    }
    (metrics_dir / 'training_done_v3.json').write_text(json.dumps(variant_done, indent=2), encoding='utf-8')
    if Path(MATRIX_SUMMARY).exists():
        shutil.copy2(MATRIX_SUMMARY, metrics_dir / 'litert_matrix_v3.json')
    if Path(RUN_STATE).exists():
        shutil.copy2(RUN_STATE, metrics_dir / 'run_state_v3_litert_matrix.json')
    if Path(RUN_EVENTS).exists():
        shutil.copy2(RUN_EVENTS, metrics_dir / 'run_events_v3_litert_matrix.jsonl')
    if Path(DISCONNECT_POINTS).exists():
        shutil.copy2(DISCONNECT_POINTS, metrics_dir / 'disconnect_points_v3_litert_matrix.jsonl')
    log_path = Path(item.get('log_path', ''))
    if log_path.exists():
        shutil.copy2(log_path, metrics_dir / f'{ARTIFACT_PREFIX}-{name}-export.log')

    with zipfile.ZipFile(bundle_zip, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
        for path in bundle_root.rglob('*'):
            if path.is_file():
                zf.write(path, path.relative_to(bundle_root).as_posix())

    bundle_result = {
        'name': name,
        'bundle_zip': str(bundle_zip),
        'bundle_size_bytes': bundle_zip.stat().st_size,
        'model_size_bytes': artifact.stat().st_size,
    }
    bundle_results.append(bundle_result)
    record_event('4_package', 'bundle_written', **bundle_result)

write_run_state(phase='4_package_done', bundles=bundle_results)
print('=== Bundles ===')
for item in bundle_results:
    print('-', item['name'])
    print('  zip:', item['bundle_zip'])
    print('  size bytes:', item['bundle_size_bytes'])
    print('  Windows finalize:')
    print(r'  cd /d D:\GemmaFit')
    print(f'  python finetune\\prepare_litert_artifact.py --source-bundle "C:\\path\\to\\{Path(item["bundle_zip"]).name}" --run-smoke')

## 5. Pixel smoke commands after local finalize

After downloading one bundle and running the Windows finalize command, push/install/test from `D:\GemmaFit`:

```powershell
adb push models\gemmafit-v3-evidence-router.litertlm /sdcard/Android/data/com.gemmafit/files/gemmafit-v3-evidence-router.litertlm
.\gradlew.bat assembleDebug --console=plain
adb install -r app\build\outputs\apk\debug\app-debug.apk
adb shell am force-stop com.gemmafit
adb logcat -c
adb shell "timeout 90 content read --uri content://com.gemmafit.debug/litert_smoke" > debug_litert_smoke_variant.json
adb logcat -d -v time > debug_litert_smoke_variant_logcat.txt
```

Interpretation:

- `init_attempts[*].stage == engine_initialize` and both GPU/CPU fail: conversion/runtime artifact issue.
- `error` contains `Input token ids are too long`: prompt/tool schema too large for `max_num_tokens`.
- `error == litert_lm_no_valid_tool_call`: engine works; tune prompt/tool schema/model behavior.
- `success == true`: artifact is Android-runnable; proceed to summary coach acceptance.